# Data Cleaning

In [16]:
import re
import warnings
import logging


import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

In [17]:
RAW_DATA_PATH= "./data.csv"
CLEANING_LOG_PATH="cleaning_log.csv"
CLEANING_LOGGING_PATH="cleaning.log"


DISTANCE_COLUMNS = [
    "dist_nearest_school_km", "dist_nearest_hospital_km",
    "dist_nearest_supermarket_km", "dist_nearest_mall_km",
    "dist_nearest_transit_station_km", "dist_nearest_cafe_restaurant_km",
]

COUNT_COLUMNS = [
    "school_count_within_3km", "hospital_count_within_3km",
    "supermarket_count_within_3km", "mall_count_within_3km",
    "transit_station_count_within_3km", "cafe_restaurant_count_within_3km",
]

# Prepare for logging

In [18]:
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler(CLEANING_LOGGING_PATH),   # writes to file
        logging.StreamHandler(),               # prints to console
    ],
)

cleaning_log: list[dict] = []

In [19]:
def log_cleaning_action(step: str,rule: str,records_affected: int,action: str,rationale: str) -> None:
    """
    Append one cleaning decision to the in-memory cleaning log.

    Args:
        step:The cleaning dimension
        rule:specific rule applied
        records_affected: Number of rows or values changed.
        action: action that was done.
        rationale: Why this action was chosen.
    """
    cleaning_log.append({
        "step": step,
        "rule": rule,
        "records_affected": records_affected,
        "action": action,
        "rationale": rationale,
    })
    logging.info(f"[LOG] {step} | {rule} | {records_affected} records | {action}")

### Load Raw Data

In [24]:
raw_df = pd.read_csv(RAW_DATA_PATH,low_memory=False)

# Work on a copy, raw_df is never modified
df=raw_df.copy()

print("DATA SHAPE")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print()

print("COLUMN NAMES")
print(df.columns.tolist())
print()

print("DTTYPES")
print(df.dtypes.value_counts())

DATA SHAPE
Rows: 37,003 | Columns: 70

COLUMN NAMES
['listing_id', 'internal_id', 'category', 'listing_type', 'detail_url', 'property_type', 'offering_type', 'completion_status', 'title', 'price_egp', 'price_period', 'price_currency', 'location_full', 'city', 'town', 'district', 'subdistrict', 'lat', 'lon', 'bedrooms', 'bathroom', 'area_value', 'area_unit', 'furnished', 'listing_level', 'is_premium', 'is_verified', 'is_featured', 'is_new_construction', 'is_direct_from_developer', 'is_exclusive', 'listed_date', 'images_count', 'has_video', 'video_url', 'reference', 'rera', 'description', 'amenities', 'payment_plan', 'agent_id', 'agent_name', 'agent_email', 'agent_is_verified', 'agent_languages', 'broker_id', 'broker_name', 'broker_email', 'broker_phone', 'contact_phone', 'contact_whatsapp', 'contact_email', 'scraped_at', 'source', 'id', 'url', 'title.1', 'scraped_at.1', 'dist_nearest_school_km', 'school_count_within_3km', 'dist_nearest_hospital_km', 'hospital_count_within_3km', 'dist_ne

# we will remove irrelevant columns to our problem before starting

In [25]:
irrelevant_columns_to_remove =[
    "listing_id","internal_id","detail_url","title","images_count",	"has_video","video_url","reference","description",
    "agent_id","agent_name","agent_email","agent_is_verified","agent_languages","broker_id","broker_name","broker_email",
    "broker_phone","contact_phone","contact_whatsapp","location_full","contact_email","scraped_at","source","id","url","title.1","scraped_at.1",
    "category", "listing_type","listed_date","subdistrict", "payment_plan"
]

print("Original number of Columns:" f"{df.shape[1]}")

df_relevance = df.drop(columns=irrelevant_columns_to_remove,errors="ignore")
print("Number of Columns after cleaning:" f"{df_relevance.shape[1]}")

Original number of Columns:70
Number of Columns after cleaning:37


In [26]:
df=df_relevance.copy()
df.columns

Index(['property_type', 'offering_type', 'completion_status', 'price_egp',
       'price_period', 'price_currency', 'city', 'town', 'district', 'lat',
       'lon', 'bedrooms', 'bathroom', 'area_value', 'area_unit', 'furnished',
       'listing_level', 'is_premium', 'is_verified', 'is_featured',
       'is_new_construction', 'is_direct_from_developer', 'is_exclusive',
       'rera', 'amenities', 'dist_nearest_school_km',
       'school_count_within_3km', 'dist_nearest_hospital_km',
       'hospital_count_within_3km', 'dist_nearest_supermarket_km',
       'supermarket_count_within_3km', 'dist_nearest_mall_km',
       'mall_count_within_3km', 'dist_nearest_transit_station_km',
       'transit_station_count_within_3km', 'dist_nearest_cafe_restaurant_km',
       'cafe_restaurant_count_within_3km'],
      dtype='object')

## Step 1: Accuracy

### Rule-based Accuracy Corrections

In [27]:
df_cleaned = df.copy()
incorrect_bedrooms_values_count=df[df["bedrooms"]=="studio"].shape[0]
df_cleaned["bedrooms"] = df["bedrooms"].replace("studio", 1).astype(int)

log_cleaning_action(step="Accuracy",rule="Bedrooms: 'studio' replaced with 1",
                    records_affected=incorrect_bedrooms_values_count,action="replaced bedrooms with studio value with 1",
                    rationale="Studio apartments typically have 1 bedroom equivalent")

INFO:root:[LOG] Accuracy | Bedrooms: 'studio' replaced with 1 | 329 records | replaced bedrooms with studio value with 1


In [28]:
df=df_cleaned.copy()

### Quarantine Invalid Records

In [29]:
# create masks
area_mask = df["area_value"] > 1000
price_mask = df["price_egp"] > 20_000_000
lon_mask = (df["lon"] < 25.0) | (df["lon"] > 35.0)
lat_mask = (df["lat"] < 22.0) | (df["lat"] > 31.0)


rejection_mask = (
    area_mask |     # unrealistic area
    price_mask |    # unrealistic price
    lon_mask |  # invalid longitude
    lat_mask    # invalid latitude
)

df_cleaned =df[~rejection_mask].copy()
df_quarantined =df[rejection_mask].copy()


if area_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="area_value > 1000",records_affected=area_mask.sum(),
        action=f"quarantined non realistic {area_mask.sum()} records",
        rationale="non realistic area values likely indicate data entry errors"
    )

if price_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="price_egp > 20_000_000",records_affected=price_mask.sum(),
        action=f"quarantined non realistic {price_mask.sum()} records",
        rationale="non realistic price values likely indicate data entry errors"
    )

if lon_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="lon out of range",records_affected=lon_mask.sum(),
        action=f"quarantined invalid {lon_mask.sum()} records",
        rationale="longitude values outside of Egypt's geographic range"
    )

if lat_mask.sum() > 0:
    log_cleaning_action(step="accuracy",rule="lat out of range",records_affected=lat_mask.sum(),
        action=f"quarantined invalid {lat_mask.sum()} records",
        rationale="latitude values outside of Egypt's geographic range"
    )    


df_quarantined["rejection_reason"] = (
    df[rejection_mask].apply(
        lambda row: '; '.join([
            'area_value > 1000' if row['area_value'] > 1000 else '',
            'price_egp > 20M' if row['price_egp'] > 20_000_000 else '',
            'invalid_lon' if (row['lon'] < 25.0 or row['lon'] > 35.0) else '',
            'invalid_lat' if (row['lat'] < 22.0 or row['lat'] > 31.0) else '',
        ]).strip('; '),
        axis=1
    )
)

log_cleaning_action(step="accuracy",rule="total_quarantined",records_affected=rejection_mask.sum(),
    action=f"quarantined {rejection_mask.sum()} records with reasons logged",
    rationale="Allows for later review and potential recovery of records if needed"
)


INFO:root:[LOG] accuracy | area_value > 1000 | 3 records | quarantined non realistic 3 records
INFO:root:[LOG] accuracy | price_egp > 20_000_000 | 876 records | quarantined non realistic 876 records
INFO:root:[LOG] accuracy | lat out of range | 1632 records | quarantined invalid 1632 records
INFO:root:[LOG] accuracy | total_quarantined | 2480 records | quarantined 2480 records with reasons logged


In [30]:
df = df_cleaned.copy()

# consistency step

In [31]:
def show_value_counts(df,columns=None):
    for col in columns:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False))

### property_type

In [32]:
df["property_type"].value_counts()

property_type
Apartments    24620
Apartment      9903
Name: count, dtype: int64

In [33]:
mapping = {
    "apartments": "apartment",
}   
df["property_type"]=df["property_type"].str.lower().replace(mapping)

log_cleaning_action(step="consistency",rule="standardize_property_types",records_affected=mapping["apartments"],
    action=f"standardized {mapping['apartments']} records from 'apartments' to 'apartment'",
    rationale="for consistency, 'apartment' is the more common term and helps unify the category"
)


INFO:root:[LOG] consistency | standardize_property_types | apartment records | standardized apartment records from 'apartments' to 'apartment'


**if is just one value drop the column**

In [34]:
if df["property_type"].nunique() ==1:
    df.drop(columns=["property_type"],inplace=True)

    log_cleaning_action(step="consistency",rule="drop_property_type_if_single_value",records_affected=0,
        action=f"dropped 'property_type' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )


INFO:root:[LOG] consistency | drop_property_type_if_single_value | 0 records | dropped 'property_type' column as it has only one unique value


### offering_type
-   for-sale                
-   Residential for Sale

convert to :<br>
-   for-sale

In [35]:
df["offering_type"].value_counts()

offering_type
for-sale                24620
Residential for Sale     9903
Name: count, dtype: int64

In [36]:
mapping = {
    'Residential for Sale': 'for-sale',
    'for-sale': 'for-sale'
}

df["offering_type"]=df["offering_type"].replace(mapping)

log_cleaning_action(step="consistency",rule="standardize_offering_types",records_affected=mapping['Residential for Sale'],
    action=f"standardized {mapping['Residential for Sale']} records from 'Residential for Sale' to 'for-sale'",
    rationale="for consistency, 'for-sale' is more concise and unifies the category"
)

INFO:root:[LOG] consistency | standardize_offering_types | for-sale records | standardized for-sale records from 'Residential for Sale' to 'for-sale'


In [37]:
df["offering_type"].value_counts()

offering_type
for-sale    34523
Name: count, dtype: int64

In [38]:
if df["offering_type"].nunique() ==1:
    df.drop(columns=["offering_type"],inplace=True)

    log_cleaning_action(step="consistency",rule="drop_offering_type_if_single_value",records_affected=0,
        action=f"dropped 'offering_type' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )


INFO:root:[LOG] consistency | drop_offering_type_if_single_value | 0 records | dropped 'offering_type' column as it has only one unique value


### completion_status
- completed             
- under-construction     
- off_plan_primary       
- completed_primary      
- off_plan                

convert to :<br>
- completed
- under-construction
- off_plan


In [39]:

mapping = {
    'completed_primary': 'completed',
    'off_plan_primary': 'off_plan'
}

df["completion_status"]=df["completion_status"].copy().replace(mapping)

log_cleaning_action(step="consistency",rule="standardize_completion_status",records_affected=mapping['off_plan_primary'],
    action=f"standardized {mapping['off_plan_primary']} records from 'off_plan_primary' to 'off_plan'",
    rationale="for consistency, 'off_plan' is more concise and unifies the category"    
)


INFO:root:[LOG] consistency | standardize_completion_status | off_plan records | standardized off_plan records from 'off_plan_primary' to 'off_plan'


In [40]:
df["completion_status"].value_counts()

completion_status
completed             22568
under-construction     8876
off_plan               3075
Name: count, dtype: int64

### price_period
**one unique value so no need to keep it**

In [41]:
df["price_period"].value_counts()

if df["price_period"].nunique() ==1:
    df.drop(columns=["price_period"],inplace=True)

    log_cleaning_action(step="consistency",rule="drop_price_period_if_single_value",records_affected=0,
        action=f"dropped 'price_period' column as it has only one unique value",    
        rationale="for consistency, the column is redundant when it contains only a single value"
    )

INFO:root:[LOG] consistency | drop_price_period_if_single_value | 0 records | dropped 'price_period' column as it has only one unique value


### price_currency
**one unique value so no need to keep it**

In [42]:
df["price_currency"].value_counts()
if df["price_currency"].nunique() ==1:
    df.drop(columns=["price_currency"],inplace=True)
    log_cleaning_action(step="consistency",rule="drop_price_currency_if_single_value",records_affected=0,
        action=f"dropped 'price_currency' column as it has only one unique value",  
        rationale="for consistency, the column is redundant when it contains only a single value"
    )
    

INFO:root:[LOG] consistency | drop_price_currency_if_single_value | 0 records | dropped 'price_currency' column as it has only one unique value


### town

In [43]:
df["town"].nunique()

95

In [44]:
df["town"].value_counts()

town
New Cairo                       7443
New Cairo City                  3758
Sheikh Zayed                    3409
6th of October                  2806
New Capital City                2147
                                ... 
Hay Helwan                         1
Cairo Alexandria Desert Road       1
Al Salam City                      1
Markaz Al Hamam                    1
Garden City                        1
Name: count, Length: 95, dtype: int64

In [45]:
# remove the word city in town names
df["town"] = df["town"].str.replace(r'\bCity\b', '', regex=True).str.strip().str.title()

log_cleaning_action(step="consistency",rule="standardize_town_names",records_affected=df["town"].str.contains(r'\bCity\b', regex=True).sum(),
    action=f"standardized town names by removing 'City' suffix and applying title case",
    rationale="for consistency, removing 'City' suffix unifies town names and title case improves readability"
)


INFO:root:[LOG] consistency | standardize_town_names | 0 records | standardized town names by removing 'City' suffix and applying title case


### district

In [46]:


# lowercase + remove extra spaces
df["district"] = (
    df["district"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

log_cleaning_action(step="consistency",rule="standardize_district_names",records_affected=df["district"].notna().sum(),
    action=f"standardized district names by lowercasing and removing extra spaces",
    rationale="for consistency, lowercasing and removing extra spaces unifies district names"
)

df["district"] = (
    df["district"]
    .str.replace(r"[^\w\s]", "", regex=True)   # remove punctuation
)

log_cleaning_action(step="consistency",rule="remove_punctuation_from_districts",records_affected=df["district"].notna().sum(),
    action=f"removed punctuation from district names",  
    rationale="for consistency, removing punctuation unifies district names"
)

INFO:root:[LOG] consistency | standardize_district_names | 34523 records | standardized district names by lowercasing and removing extra spaces
INFO:root:[LOG] consistency | remove_punctuation_from_districts | 34523 records | removed punctuation from district names


In [47]:
df['district'].nunique()

739

### area_unit

In [48]:
df['area_unit'].value_counts()

area_unit
SQM    24620
sqm     9903
Name: count, dtype: int64

In [49]:
df['area_unit'] = df['area_unit'].str.lower().str.strip()

log_cleaning_action(step="consistency",rule="standardize_area_unit",records_affected=df['area_unit'].notna().sum(),
    action=f"standardized 'area_unit' by lowercasing and stripping spaces",
    rationale="for consistency, lowercasing and stripping spaces unifies area unit values"
)


INFO:root:[LOG] consistency | standardize_area_unit | 34523 records | standardized 'area_unit' by lowercasing and stripping spaces


In [50]:
if df['area_unit'].nunique() ==1:
    df.drop(columns=['area_unit'],inplace=True)
    log_cleaning_action(step="consistency",rule="drop_area_unit_if_single_value",records_affected=0,
        action=f"dropped 'area_unit' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )

INFO:root:[LOG] consistency | drop_area_unit_if_single_value | 0 records | dropped 'area_unit' column as it has only one unique value


### furnished

In [51]:
df['furnished'].value_counts()

furnished
unfurnished    20143
NO              4522
furnished       1366
PARTLY           922
YES              473
Name: count, dtype: int64

In [52]:
mapping = {'NO': 'unfurnished', 'YES': 'furnished','PARTLY':'partly'}
df['furnished'] = df['furnished'].replace(mapping)
log_cleaning_action(step="consistency",rule="standardize_furnished",records_affected=df['furnished'].isin(mapping.keys()).sum(),
    action=f"standardized 'furnished' values by mapping {mapping}",
    rationale="for consistency, unifying furnished status values improves clarity and analysis"
)

INFO:root:[LOG] consistency | standardize_furnished | 0 records | standardized 'furnished' values by mapping {'NO': 'unfurnished', 'YES': 'furnished', 'PARTLY': 'partly'}


In [53]:
df['furnished'].value_counts()

furnished
unfurnished    24665
furnished       1839
partly           922
Name: count, dtype: int64

### col with one unique value
- is_verified 
-  is_new_construction 
-  rera

**are only one unique value so we can drop them**

In [54]:
print(df["is_new_construction"].value_counts())
print(df["is_verified"].value_counts())
print(df["rera"].value_counts())

is_new_construction
False    34523
Name: count, dtype: int64
is_verified
False    34523
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


In [55]:
if df["is_verified"].nunique() ==1:
    df.drop(columns=['is_verified'], inplace=True)  

    log_cleaning_action(step="consistency",rule="drop_is_verified_if_single_value",records_affected=0,
        action=f"dropped 'is_verified' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )
    
if df["is_new_construction"].nunique() ==1:
    df.drop(columns=['is_new_construction'], inplace=True) 

    log_cleaning_action(step="consistency",rule="drop_is_new_construction_if_single_value",records_affected=0,
        action=f"dropped 'is_new_construction' column as it has only one unique value",
        rationale="for consistency, the column is redundant when it contains only a single value"
    )

    
if df["rera"].nunique() <=1:
    df.drop(columns=['rera'], inplace=True) 
    log_cleaning_action(step="consistency",rule="drop_rera_if_single_or_no_value",records_affected=0,
        action=f"dropped 'rera' column as it has only one or no unique value",
        rationale="for consistency, the column is redundant when it contains only a single or no value"
    ) 

INFO:root:[LOG] consistency | drop_is_verified_if_single_value | 0 records | dropped 'is_verified' column as it has only one unique value
INFO:root:[LOG] consistency | drop_is_new_construction_if_single_value | 0 records | dropped 'is_new_construction' column as it has only one unique value
INFO:root:[LOG] consistency | drop_rera_if_single_or_no_value | 0 records | dropped 'rera' column as it has only one or no unique value


### amenities

In [56]:
for unique_value in df['amenities'].unique():
    print({unique_value})

{"Balcony | Built in Wardrobes | Central A/C | Covered Parking | Shared Pool | Security | Shared Spa | Shared Gym | View of Landmark | Lobby in Building | Children's Pool"}
{"Balcony | Built in Wardrobes | Central A/C | Covered Parking | Kitchen Appliances | Maids Room | Private Garden | Private Pool | Shared Pool | Study | View of Water | Security | Shared Spa | Shared Gym | Walk-in Closet | View of Landmark | Lobby in Building | Children's Pool"}
{'Balcony | Central A/C | Built in Wardrobes | Security'}
{'Private Garden | Covered Parking | Balcony | Built in Wardrobes | Central A/C | Maids Room | Kitchen Appliances | Shared Pool | Private Pool | View of Water | Study'}
{"Covered Parking | Shared Pool | Study | Security | Shared Spa | Shared Gym | Children's Pool"}
{'Balcony | View of Landmark | Shared Gym | Security'}
{'Balcony | Built in Wardrobes | Covered Parking | Shared Pool | Security | Walk-in Closet | View of Landmark | Lobby in Building'}
{"Balcony | Built in Wardrobes | Kit

In [57]:
import pandas as pd
import numpy as np

def standardize_amenities(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

   
    x = x.strip("{}")
    x = x.strip('"')
    x = x.strip("'")

   
    amenities = [item.strip().lower() for item in x.split("|")]

  
    amenities = [a for a in amenities if a]

    
    amenities = sorted(set(amenities))

    
    return " | ".join(amenities)

df["amenities"] = df["amenities"].apply(standardize_amenities)

## Step 2: Consistency

### Coerce Types for Numerical , Bool and Categorical Columns

In [58]:
df = df.replace(
    ["nan", "none", "None", "", "null", "NULL"],
    np.nan
)
numeric_cols = [
    "price_egp",
    "lat",
    "lon",
    "area_value",
    "dist_nearest_school_km",
    "school_count_within_3km",
    "dist_nearest_hospital_km",
    "hospital_count_within_3km",
    "dist_nearest_supermarket_km",
    "supermarket_count_within_3km",
    "dist_nearest_mall_km",
    "mall_count_within_3km",
    "dist_nearest_transit_station_km",
    "transit_station_count_within_3km",
    "dist_nearest_cafe_restaurant_km",
    "cafe_restaurant_count_within_3km"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce").astype("Int64")
df["bathroom"] = (
    df["bathroom"]
    .replace({
        "none": np.nan,
        "7+": 7
    })
)

df["bathroom"] = pd.to_numeric(
    df["bathroom"],
    errors="coerce"
).astype("Int64")

bool_cols = [
    "is_premium",
    "is_featured",
    "is_direct_from_developer",
    "is_exclusive"
]

for col in bool_cols:
    df[col] = df[col].astype("boolean")

In [59]:
df["lat"] = df["lat"].round(6)
df["lon"] = df["lon"].round(6)

In [62]:
cat_cols = [
    "completion_status",
    "city",
    "town",
    "district",
    "furnished",
    "listing_level",
    "amenities",
]

for col in cat_cols:
    df[col] = df[col].astype("category")

In [63]:
df_consistency = df.copy()

df=df_consistency.copy()

## Step 3: Completeness

### Reporting Missingness

In [65]:
def replace_placeholder_strings(dataframe: pd.DataFrame, columns: list) -> int:
    """
    Replace placeholder strings in specified columns with NaN.
    
    Args:
        dataframe: DataFrame to clean
        columns: List of column names to process
        
    Returns:
        Total count of replacements made
    """

    PLACEHOLDER_STRINGS = [
        "", " ", "?", "N/A", "n/a", "Unknown", "unknown",
        "none", "None", "null", "Null", "missing", "Missing", "-",
    ]


    total_replacements = 0
    for col in columns:
        mask = dataframe[col].astype(str).str.strip().isin(PLACEHOLDER_STRINGS)
        replacements = mask.sum()
        dataframe.loc[mask, col] = np.nan
        total_replacements += replacements
    return total_replacements

# Apply to all string columns
string_cols = df.select_dtypes(include=["object"]).columns.tolist()
replaced_count = replace_placeholder_strings(df, string_cols)

In [66]:
def audit_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Produce a full missing value report sorted by missingness descending.

    Args:
        df: Input DataFrame.

    Returns:
        DataFrame with columns: column, missing_count, missing_pct, dtype.
    """
    missing_counts = df.isna().sum()
    missing_pcts   = (df.isna().mean() * 100).round(2)

    report = pd.DataFrame({
        "column":        missing_counts.index,
        "missing_count": missing_counts.values,
        "missing_pct":   missing_pcts.values,
        "dtype":         df.dtypes.values,
    })

    return report[report["missing_count"] > 0].sort_values("missing_pct", ascending=False)


missing_report = audit_missing_values(df)
print(f"Columns with missing values: {len(missing_report)}")
missing_report

Columns with missing values: 6


,column,missing_count,missing_pct,dtype
15,is_exclusive,24620,71.31,boolean
16,amenities,24620,71.31,category
10,furnished,7097,20.56,category
4,district,4485,12.99,category
0,completion_status,4,0.01,category
8,bathroom,2,0.01,Int64


In [ ]:
df["furnished"].value_counts(dropna=False)

what should we do in amentities

In [67]:
cols_to_drop = ["is_exclusive",  # about 80% missing and value exist in listing_level column which is more complete
                ]
df = df.drop(columns=cols_to_drop)
log_cleaning_action(
    step="Completeness",
    rule=f"Drop {', '.join(cols_to_drop)} cols high-missingness",
    records_affected=len(df),
    action=f"Dropped columns: {cols_to_drop}",
    rationale="Columns with >50% missing AND not required as model features",
)
print(f"Shape after column drops: {df.shape}")
print(f"Dropped {len(cols_to_drop)} columns")

INFO:root:[LOG] Completeness | Drop is_exclusive cols high-missingness | 34523 records | Dropped columns: ['is_exclusive']


Shape after column drops: (34523, 28)
Dropped 1 columns


In [68]:
def fill_district_with_mode(df):
    group_mode = df.groupby(['city', 'town'])['district'].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
    df['district'] = df['district'].fillna(group_mode)

    city_mode = df.groupby('city')['district'].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
    df['district'] = df['district'].fillna(city_mode)

    df['district'] = df['district'].fillna('Unknown')

    return df

In [ ]:
fill_district_with_mode(df)

CategoricalDtype(categories=['furnished', 'partly', 'unfurnished'], ordered=False, categories_dtype=object)

In [69]:
if "furnished" in df.columns:
    n_furnished_missing = int(df["furnished"].isna().sum())
    df["furnished"] = df["furnished"].fillna("Unknown")
    log_cleaning_action(
        step="Completeness",
        rule="furnished: fill NaN with 'Unknown' category",
        records_affected=n_furnished_missing,
        action="fillna('Unknown')",
        rationale=(
            "Missingness is informative — agents who don't disclose furnished status "
            "may represent a distinct listing pattern. 'Unknown' lets the model learn from it."
        ),
    )

TypeError: Cannot setitem on a Categorical with a new category (Unknown), set the categories first

In [72]:
number_missing_completion = int(df["completion_status"].isna().sum())
df = df.dropna(subset=['completion_status'])
log_cleaning_action(
    step="Completeness",
    rule="Drop rows with missing completion status",
    records_affected=number_missing_completion,
    action="dropna(subset=['completion'])",
    rationale=(
        "Completion status is a critical feature for modeling. "
        "Missingness is relatively low, so dropping is preferable to imputation since very few rows are affected."
    ),
)

INFO:root:[LOG] Completeness | Drop rows with missing completion status | 4 records | dropna(subset=['completion'])


In [73]:
number_missing_bathrooms = int(df["bathroom"].isna().sum())
df = df.dropna(subset=['bathroom'])
log_cleaning_action(
    step="Completeness",
    rule="Drop rows with missing bathroom count",
    records_affected=number_missing_bathrooms,
    action="dropna(subset=['bathroom'])",
    rationale=(
        "Bathroom count is a critical feature for modeling. "
        "Missingness is relatively low, so dropping is preferable to imputation since very few rows are affected."
    ),
)

INFO:root:[LOG] Completeness | Drop rows with missing bathroom count | 2 records | dropna(subset=['bathroom'])


### Imputing Numeric Columns

In [ ]:
NUMERIC_MEDIAN_IMPUTE_COLS = ["bedrooms", "bathrooms", "area_value"]


def impute_numeric_median(
    df: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """Impute missing values in numeric columns using the column median.

    Use for: normally distributed or mildly skewed numeric features
    with <5% missingness and MCAR pattern.
    """
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            continue
        n_missing = int(df[col].isna().sum())
        if n_missing == 0:
            continue
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        log_cleaning_action(
            step="Completeness",
            rule=f"{col}: median imputation",
            records_affected=n_missing,
            action=f"Fill NaN with median ({median_val:.2f})",
            rationale="MCAR pattern, <5% missing, critical feature — median robust to skew",
        )
    return df


df = impute_numeric_median(df, NUMERIC_MEDIAN_IMPUTE_COLS)

# Re-cast bedrooms and bathrooms to Int64 after median fill 
df["bedrooms"] = df["bedrooms"].round().astype("Int64")
df["bathrooms"] = df["bathrooms"].round().astype("Int64")

print(df[[c for c in NUMERIC_MEDIAN_IMPUTE_COLS if c in df.columns]].isna().sum())

### Imputing District Column

In [ ]:
def impute_district_by_location_group(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing district values using forward/backward fill within city-town groups.

    Logic:
    1. Within each (city, town) group, propagate known district values forward
       then backward to fill gaps.
    2. Any remaining NaN (entire group has no known district) -> fill with "Unknown".
    """
    df = df.copy()
    if "district" not in df.columns:
        print("No 'district' column found; skipping district imputation.")
        return df

    n_missing_before = int(df["district"].isna().sum())

    if ("city" in df.columns) and ("town" in df.columns):
        df["district"] = (
            df.groupby(["city", "town"])['district']
              .transform(lambda x: x.ffill().bfill())
        )
        n_filled_by_group = n_missing_before - int(df["district"].isna().sum())
    else:
        n_filled_by_group = 0

    # Remaining NaN -> "Unknown"
    df["district"] = df["district"].fillna("Unknown")

    log_cleaning_action(
        step="Completeness",
        rule="district: grouped ffill/bfill within city+town, then 'Unknown'",
        records_affected=n_missing_before,
        action=f"{n_filled_by_group} filled by group; rest -> 'Unknown'",
        rationale="MAR pattern: missingness depends on city/town, not the district value itself",
    )
    return df


df = impute_district_by_location_group(df)
print(f"district nulls remaining: {df['district'].isna().sum()}")

### Imputing Bool Columns

In [ ]:
def add_missing_flag_and_impute_mode(
    df: pd.DataFrame,
    column: str,
    group_by: list[str] | None = None,
) -> pd.DataFrame:
    """Impute missing values with mode."""
    df = df.copy()
    n_missing = int(df[column].isna().sum())

    if group_by and all(col in df.columns for col in group_by):
        df[column] = df.groupby(group_by)[column].transform(
            lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
        )

    # Global fallback
    global_mode = df[column].mode()
    if not global_mode.empty:
        df[column] = df[column].fillna(global_mode[0])

    log_cleaning_action(
        step="Completeness",
        rule=f"{column}: mode imputation (group_by={group_by})",
        records_affected=n_missing,
        action="Filled with mode",
        rationale="Moderate missingness, the mode preserves the dominant pattern",
    )
    return df


# completion_status: impute by city mode (if column exists)
if "completion_status" in df.columns:
    df = add_missing_flag_and_impute_mode(df, "completion_status", group_by=["city"] if "city" in df.columns else None)

# furnished: create explicit 'Unknown' category
if "furnished" in df.columns:
    n_furnished_missing = int(df["furnished"].isna().sum())
    df["furnished"] = df["furnished"].fillna("Unknown")
    log_cleaning_action(
        step="Completeness",
        rule="furnished: fill NaN with 'Unknown' category",
        records_affected=n_furnished_missing,
        action="fillna('Unknown')",
        rationale=(
            "Missingness is informative — agents who don't disclose furnished status "
            "may represent a distinct listing pattern. 'Unknown' lets the model learn from it."
        ),
    )

print(df[["completion_status", "furnished"]].isna().sum())

In [ ]:
print(f"\nTotal nulls remaining across all columns:")
print(df.isna().sum()[df.isna().sum() > 0])

### Completeness Step Summary

In [ ]:
print("COMPLETENESS CLEANING SUMMARY\n")
remaining_nulls = df.isna().sum()
columns_with_nulls = remaining_nulls[remaining_nulls > 0]
if columns_with_nulls.empty:
    print("No missing values remain in the dataset.")
else:
    print(f"Columns still with missing values: {len(columns_with_nulls)}")
    print(columns_with_nulls)
print(f"\nFinal shape: {df.shape}")

## Step 4: Uniqueness

### Removing Duplicates

In [ ]:
def remove_exact_duplicates(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Remove exact duplicate rows.

    Args:
        df: Input DataFrame.
        subset: Columns to use as the duplicate key.

    Returns:
        Tuple of (df_deduped, df_removed_duplicates).
    """
    valid_subset = df.columns.to_list()

    duplicate_mask = df.duplicated(subset=valid_subset)
    df_removed = df[duplicate_mask].copy()
    df_deduped = df[~duplicate_mask].copy()

    log_cleaning_action(
        step="Uniqueness",
        rule=f"Exact duplicates on subset: {valid_subset}",
        records_affected=int(duplicate_mask.sum()),
        action=f"Keep first occurence. duplicates saved to interim/",
        rationale="Identical portal scrapes of the same listing inflate training data",
    )
    return df_deduped, df_removed

n_before = len(df)
df, df_removed_duplicates = remove_exact_duplicates(df)
n_after = len(df)

print(f"Records before dedup: {n_before:,}")
print(f"Records removed:      {n_before - n_after:,}")
print(f"Records remaining:    {n_after:,}")

# Save removed duplicates to interim for audit
Path("data/interim").mkdir(parents=True, exist_ok=True)
df_removed_duplicates.to_csv("data/interim/removed_duplicates.csv", index=False)
print("Saved removed duplicates to data/interim/removed_duplicates.csv")

### Uniquess Step Summary

In [ ]:
print("UNIQUENESS CLEANING SUMMARY\n")
print(f"Exact duplicates removed:           {len(df_removed_duplicates):,}")
print(f"Records after deduplication:        {len(df):,}")

## Step 5: Outliers

### Detect Outliers Using IQR

In [ ]:
def detect_outliers_iqr(
    series: pd.Series,
    multiplier: float = IQR_MULTIPLIER,
) -> pd.Series:
    """Return a boolean mask: True where the value is an IQR outlier.

    Args:
        series: Numeric pandas Series to check.
        multiplier: IQR multiplier for fence calculation (default 1.5).

    Returns:
        Boolean Series, True where value is outside [Q1 - m*IQR, Q3 + m*IQR].
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - multiplier * iqr
    upper_fence = q3 + multiplier * iqr
    return (series < lower_fence) | (series > upper_fence)


outlier_target_cols = ["price_egp", "area_value"] + DISTANCE_COLUMNS + COUNT_COLUMNS
blocked_columns = set(globals().get("cols_to_drop", []))
valid_outlier_cols = [c for c in outlier_target_cols if c in df.columns and c not in blocked_columns]
outlier_audit = {}

for col in valid_outlier_cols:
    mask = detect_outliers_iqr(df[col].dropna())
    outlier_audit[col] = int(mask.sum())

outlier_audit_df = (
    pd.DataFrame.from_dict(outlier_audit, orient="index", columns=["iqr_outliers"])
    .sort_values("iqr_outliers", ascending=False)
)
print("IQR Outlier Counts (confirming Phase 1 findings):")
outlier_audit_df

### Cap columns at percentile thresholds

In [ ]:
def cap_column(
    df: pd.DataFrame,
    column: str,
    lower_percentile: float,
    upper_percentile: float,
) -> pd.DataFrame:
    """Cap a numeric column's values at specified percentile thresholds.

    Args:
        df: Input DataFrame.
        column: Column to cap.
        lower_percentile: Lower bound percentile (e.g., 0.02 for P2).
        upper_percentile: Upper bound percentile (e.g., 0.98 for P98).

    Returns:
        DataFrame with column values clipped between computed bounds.
    """
    df = df.copy()
    lower_cap = df[column].quantile(lower_percentile)
    upper_cap = df[column].quantile(upper_percentile)
    n_below = int((df[column] < lower_cap).sum())
    n_above = int((df[column] > upper_cap).sum())
    df[column] = df[column].clip(lower=lower_cap, upper=upper_cap)
    log_cleaning_action(
        step="Outliers",
        rule=f"{column}: cap at P{int(lower_percentile*100)}–P{int(upper_percentile*100)}",
        records_affected=n_below + n_above,
        action=f"Clip to [{lower_cap:,.0f}, {upper_cap:,.0f}]",
        rationale=f"{n_below} below lower cap, {n_above} above upper cap — capping preserves dataset size",
    )
    return df


df = cap_column(df, "price_egp", PRICE_CAP_LOWER_PERCENTILE, PRICE_CAP_UPPER_PERCENTILE)
print(f"price_egp range after capping: {df['price_egp'].min():,.0f} – {df['price_egp'].max():,.0f}")

In [ ]:
df = cap_column(df, "area_value", AREA_CAP_LOWER_PERCENTILE, AREA_CAP_UPPER_PERCENTILE)

# Re-run cross-field validation: after capping, some records may still violate
# the bedroom/area ratio rule. Quarantine these as well.
if "bedrooms" in df.columns and "area_value" in df.columns:
    if "df_quarantine" not in globals():
        df_quarantine = pd.DataFrame(columns=list(df.columns) + ["rejection_reason"])

    area_bedroom_violation = (
        df["bedrooms"].astype(float, errors="ignore") >= 3
    ) & (df["area_value"] < (df["bedrooms"].astype(float, errors="ignore") * MIN_AREA_PER_BEDROOM))

    n_new_violations = int(area_bedroom_violation.sum())
    if n_new_violations > 0:
        new_quarantine = df[area_bedroom_violation].copy()
        new_quarantine["rejection_reason"] = "area_bedroom_ratio_implausible_post_capping"
        df_quarantine = pd.concat([df_quarantine, new_quarantine], ignore_index=True)
        df = df[~area_bedroom_violation].copy()
        log_cleaning_action(
            step="Outliers",
            rule="Cross-field re-validation post area capping",
            records_affected=n_new_violations,
            action="Quarantined records with implausible area/bedroom ratio",
            rationale="Area capping may expose previously hidden ratio violations",
        )

print(f"area_value range after capping: {df['area_value'].min():.1f} – {df['area_value'].max():.1f} sqm")
print(f"Updated quarantine size: {len(df_quarantine):,}")

In [ ]:
def cap_columns_at_percentile(
    df: pd.DataFrame,
    columns: list[str],
    upper_percentile: float,
) -> pd.DataFrame:
    """Cap multiple columns at the same upper percentile threshold.

    Args:
        df: Input DataFrame.
        columns: List of column names to cap.
        upper_percentile: Upper bound percentile.

    Returns:
        DataFrame with all specified columns capped.
    """
    df = df.copy()
    for col in columns:
        if col not in df.columns:
            continue
        df = cap_column(df, col, lower_percentile=0.0, upper_percentile=upper_percentile)
    return df


blocked_columns = set(globals().get("cols_to_drop", []))
valid_dist_cols = [c for c in DISTANCE_COLUMNS if c in df.columns and c not in blocked_columns]
df = cap_columns_at_percentile(df, valid_dist_cols, POI_DIST_CAP_PERCENTILE)

print("Distance column ranges after capping:")
for col in valid_dist_cols:
    print(f"  {col}: {df[col].min():.3f} – {df[col].max():.3f} km")

### Outlier Step Summary

In [ ]:
print("OUTLIER TREATMENT SUMMARY\n")
print(f"price_egp:   capped at P{int(PRICE_CAP_LOWER_PERCENTILE*100)}–P{int(PRICE_CAP_UPPER_PERCENTILE*100)}")
print(f"area_value:  capped at P{int(AREA_CAP_LOWER_PERCENTILE*100)}–P{int(AREA_CAP_UPPER_PERCENTILE*100)}")
print(f"dist_* cols: capped at P{int(POI_DIST_CAP_PERCENTILE*100)}")
print(f"\nFinal shape: {df.shape}")

## Target Re-derivation

`price_category` must be re-derived from the cleaned `price_egp` after:
- Accuracy corrections (prices set to NaN or corrected)
- Price capping (P2–P98)

Using the same design from Phase 1: quantile binning into 3 equal tiers ensures ~33.3% per class.

In [ ]:
def derive_price_category(df: pd.DataFrame, price_col: str = "price_egp") -> pd.DataFrame:
    """Derive the price_category target variable from cleaned price using quantile binning.

    Bins:
    - Low:    bottom 33.3% by price
    - Medium: middle 33.3% by price
    - High:   top 33.3% by price

    Args:
        df: Input DataFrame with a clean numeric price column.
        price_col: Name of the price column.

    Returns:
        DataFrame with 'price_category' column added (dtype: category).

    Raises:
        ValueError: If price_col contains NaN values (must be imputed first).
    """
    if df[price_col].isna().any():
        raise ValueError(
            f"'{price_col}' still contains NaN. Impute before deriving target."
        )

    df = df.copy()
    df["price_category"] = pd.qcut(
        df[price_col],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop",
    )

    log_cleaning_action(
        step="Target",
        rule="price_category derived from cleaned price_egp via quantile binning",
        records_affected=len(df),
        action="pd.qcut(q=3, labels=['Low','Medium','High'])",
        rationale="Re-derive after accuracy corrections and capping to ensure consistency",
    )
    return df


# Handle any remaining NaN in price_egp before target derivation
n_price_null = int(df["price_egp"].isna().sum())
if n_price_null > 0:
    price_median = df["price_egp"].median()
    df["price_egp"] = df["price_egp"].fillna(price_median)
    log_cleaning_action(
        step="Completeness",
        rule="price_egp: median imputation for remaining nulls",
        records_affected=n_price_null,
        action=f"fillna({price_median:,.0f})",
        rationale="Required before target derivation; MCAR assumption",
    )

df = derive_price_category(df)

print("price_category distribution:")
print(df["price_category"].value_counts())
print(df["price_category"].value_counts(normalize=True).round(3))

In [ ]:
# Range checks
if "area_value" in df.columns:
    assert (df["area_value"] >= AREA_MIN_SQM).all(), "area_value below minimum"
if "price_egp" in df.columns:
    assert (df["price_egp"] > 0).all(), "price_egp has non-positive values"
if "latitude" in df.columns:
    assert df["latitude"].between(LAT_MIN, LAT_MAX).all(), "latitude out of Egypt bounds"
if "longitude" in df.columns:
    assert df["longitude"].between(LON_MIN, LON_MAX).all(), "longitude out of Egypt bounds"
print("All range assertions passed")

## Save Outputs

In [ ]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved clean data → {PROCESSED_DATA_PATH}")
print(f"Shape: {df.shape}")

In [ ]:
df_quarantine.to_csv(QUARANTINE_DATA_PATH, index=False)
print(f"Saved quarantine → {QUARANTINE_DATA_PATH}")
print(f"Quarantine shape: {df_quarantine.shape}")
# print(f"\nRejection reason breakdown:")
# print(df_quarantine["rejection_reason"].value_counts())

In [ ]:
Path("reports").mkdir(parents=True, exist_ok=True)
if cleaning_log:
    cleaning_log_df = pd.DataFrame(cleaning_log)
    cleaning_log_df.to_csv(CLEANING_LOG_PATH, index=False)
    print(f"Saved cleaning log → {CLEANING_LOG_PATH}")
    print(f"Total log entries: {len(cleaning_log_df)}")
    cleaning_log_df
else:
    print("No cleaning log entries to save")

In [ ]:
print("DATA CLEANING SUMMARY\n")
print(f"Raw records:               {len(raw_df):>8,}")
print(f"Quarantined records:       {len(df_quarantine):>8,}  ({len(df_quarantine)/len(raw_df)*100:.1f}%)")
print(f"Duplicates removed:        {len(df_removed_duplicates):>8,}  ({len(df_removed_duplicates)/len(raw_df)*100:.1f}%)")
print(f"Clean records:             {len(df):>8,}  ({len(df)/len(raw_df)*100:.1f}%)")
print()
print(f"Original columns:          {len(raw_df.columns):>8}")
print(f"Columns dropped:           {len(cols_to_drop):>8}")
print(f"New columns added:         {len(df.columns) - (len(raw_df.columns) - len(cols_to_drop)):>8}")
print(f"Final columns:             {len(df.columns):>8}")
print()
print(f"Target distribution:")
dist = df["price_category"].value_counts()
for cls in ["Low", "Medium", "High"]:
    count = dist.get(cls, 0)
    print(f"  {cls:<10}: {count:>6,}  ({count/len(df)*100:.1f}%)")
print()
print(f"Outputs saved:")
print(PROCESSED_DATA_PATH)
print(QUARANTINE_DATA_PATH)
print(CLEANING_LOG_PATH)